# H&M Transaction Data: Product Recommendations 01
## Pre-process data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder

# Define paths
base_path = Path("../data/")
raw_path = processed_path = base_path / 'raw'
processed_path = base_path / 'processed'

## Load Cleaned Data

In [2]:
customers = pd.read_csv(processed_path / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(processed_path / 'transactions_hm_cleaned.csv')
articles = pd.read_csv(processed_path / 'articles_hm_cleaned.csv')

print("Cleaned Rows of Data:")
print(f"Articles: {len(articles):,}")
print(f"Customers: {len(customers):,}")
print(f"Transactions: {len(transactions):,}")

# convert transaction date to datetime
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

Cleaned Rows of Data:
Articles: 105,542
Customers: 1,048,575
Transactions: 1,040,101


## Train, Validation Data Generation

Data builded uses a max total number of transaction
It preferentially allocates those transaction to the prediction period and uses a 1:1 ratio of prediction period transaction and pre-prediction transactions

In [3]:
HISTORY_DAYS = 30

def build_product_recommendation_data(prediction_start_date, prediction_end_date, history_days=90,random_state=67):
    as_of_date = pd.to_datetime(prediction_start_date) - pd.Timedelta(days=1)
    history_start = as_of_date - pd.Timedelta(days=history_days)
    prediction_start_date = pd.to_datetime(prediction_start_date)
    prediction_end_date = pd.to_datetime(prediction_end_date)
    
    print(f"History start date: {history_start:%Y-%m-%d}")
    print(f"Data as of date: {as_of_date:%Y-%m-%d}")
    print(f"Prediction period: {prediction_start_date:%Y-%m-%d} to {prediction_end_date:%Y-%m-%d}")

    # Transactions in prediction period
    prediction_period_mask = (transactions['t_dat'] >= prediction_start_date) & (transactions['t_dat'] <= prediction_end_date)
    transactions_prediction_period = transactions[prediction_period_mask]
    prediction_transactions_count = len(transactions_prediction_period)
    print(f"Transactions in prediction period: {len(transactions_prediction_period):,}")

    # Transactions before prediction period
    history_mask = (transactions['t_dat'] >= history_start) & (transactions['t_dat'] <= as_of_date)
    transactions_before_prediction = transactions[history_mask]
    print(f"Transaction rows before prediction: {len(transactions_before_prediction):,}")

    transactions_df = pd.concat([transactions_prediction_period, transactions_before_prediction])
    print(f"Transaction rows: {len(transactions_df):,}")
    
    # Pull all customers and articles that are in the sampled transactions
    sample_customer_ids = transactions_df['customer_id'].unique()
    sample_article_ids = transactions_df['article_id'].unique()
    
    customers_sample = customers[customers['customer_id'].isin(sample_customer_ids)]
    articles_sample = articles[articles['article_id'].isin(sample_article_ids)]
    
    customers_df = customers_sample
    articles_df = articles_sample

    
    cfe = CustomerFeatureEngineer(customers_df=customers_df, transactions_df=transactions_df)
    customer_features = cfe.calculate_all_features(articles_df=articles_df, as_of_date=as_of_date)
    print(f"Customer rows: {len(customer_features):,}")

    pfe = ProductFeatureEngineer(articles_df=articles_df, transactions_df=transactions_df)
    product_features = pfe.calculate_all_features(as_of_date=as_of_date)
    print(f"Product rows: {len(product_features):,}")

    builder = RecommendationTrainingBuilder(transactions_df=transactions_df,
                                            customer_features_df=customer_features,
                                            product_features_df=product_features,
                                            customer_fill_values=cfe.get_fill_values(),
                                            product_fill_values=pfe.get_fill_values()
                                           )
    data = builder.build_dataset(prediction_start=prediction_start_date,
                                            prediction_end=prediction_end_date,
                                            negative_ratio=5,
                                            random_state=random_state)

    print(f"Total rows: {len(data):,}")
    print(f"- Positives: {(data['purchased'] == 1).sum():,}")
    print(f"- Negatives: {(data['purchased'] == 0).sum():,}")

    unique_pairs = transactions_prediction_period[['customer_id', 'article_id']].drop_duplicates()
    missing_products = set(unique_pairs['article_id'].unique()) - set(product_features['article_id'].unique())
    missing_product_pairs = unique_pairs[unique_pairs['article_id'].isin(missing_products)].shape[0]
    missing_customers = set(unique_pairs['customer_id'].unique()) - set(customer_features['customer_id'].unique())
    print(f"Unique customer-product pairs in prediction period: {unique_pairs.shape[0]:,}")
    print(f"  - Duplicate pairs dropped: {prediction_transactions_count - unique_pairs.shape[0]:,}")
    print(f"  - Products filled with sentinels: {len(missing_products):,} products ({missing_product_pairs:,} pairs affected)")
    print(f"  - Customers filled with sentinels: {len(missing_customers):,} customers")
    print(f"  - Final positives: {(data['purchased'] == 1).sum():,}")

    return data

In [4]:
print("=========TRAINING DATA=========")
train_prediction_start_date = "2019-06-01"
train_prediction_end_date = "2019-06-15"
train_data =  build_product_recommendation_data(train_prediction_start_date, train_prediction_end_date, history_days=HISTORY_DAYS)

=========TRAINING DATA=========
History start date: 2019-05-01
Data as of date: 2019-05-31
Prediction period: 2019-06-01 to 2019-06-15
Transactions in prediction period: 47,558
Transaction rows before prediction: 98,958
Transaction rows: 146,516
Customer rows: 82,027
Product rows: 16,400
Total rows: 284,730
- Positives: 47,455
- Negatives: 237,275
Unique customer-product pairs in prediction period: 47,455
  - Duplicate pairs dropped: 103
  - Products filled with sentinels: 2,595 products (7,033 pairs affected)
  - Customers filled with sentinels: 8,440 customers
  - Final positives: 47,455


In [5]:
print("=========VALIDATION DATA=========")
val_prediction_start = "2019-06-16"
val_prediction_end = "2019-06-30"
val_data =  build_product_recommendation_data(val_prediction_start, val_prediction_end, history_days=HISTORY_DAYS)

=========VALIDATION DATA=========
History start date: 2019-05-16
Data as of date: 2019-06-15
Prediction period: 2019-06-16 to 2019-06-30
Transactions in prediction period: 73,510
Transaction rows before prediction: 102,319
Transaction rows: 175,829
Customer rows: 96,452
Product rows: 16,469
Total rows: 440,082
- Positives: 73,347
- Negatives: 366,735
Unique customer-product pairs in prediction period: 73,347
  - Duplicate pairs dropped: 163
  - Products filled with sentinels: 3,589 products (8,161 pairs affected)
  - Customers filled with sentinels: 12,560 customers
  - Final positives: 73,347


In [6]:
print("=========TEST DATA=========")
test_prediction_start = "2019-07-01"
test_prediction_end = "2019-07-15"
test_data =  build_product_recommendation_data(test_prediction_start, test_prediction_end, history_days=HISTORY_DAYS)

=========TEST DATA=========
History start date: 2019-05-31
Data as of date: 2019-06-30
Prediction period: 2019-07-01 to 2019-07-15
Transactions in prediction period: 54,597
Transaction rows before prediction: 125,420
Transaction rows: 180,017
Customer rows: 98,337
Product rows: 17,233
Total rows: 326,736
- Positives: 54,456
- Negatives: 272,280
Unique customer-product pairs in prediction period: 54,456
  - Duplicate pairs dropped: 141
  - Products filled with sentinels: 2,625 products (5,076 pairs affected)
  - Customers filled with sentinels: 9,655 customers
  - Final positives: 54,456


In [7]:
feature_cols = [col for col in train_data.columns if col not in ['customer_id', 'article_id', 'purchased']]

print(f"Feature Columns:")
for col in feature_cols:
    print(f"- {col}")


Feature Columns:
- sales_last_7_days
- sales_last_30_days
- days_since_first_sale
- days_since_last_sale
- avg_price
- min_price
- max_price
- product_price_std
- age
- customer_price_std
- num_purchases
- total_spent
- days_since_last_purchase
- avg_transaction_value
- avg_days_between_purchases
- primary_department
- primary_garment_group
- category_diversity
- is_new_customer
- FN
- Active
- club_member_status_NOT_ACTIVE_MEMBER
- club_member_status_PRE-CREATE
- fashion_news_frequency_REGULARLY


### Categorical features
One hot encode the garment groups. Drop primary department, too many possibilities

In [8]:
print(f"Unique departments: {train_data['primary_department'].nunique()}")
print(f"Unique garment groups: {train_data['primary_garment_group'].nunique()}")

Unique departments: 171
Unique garment groups: 21


In [9]:
train_data = train_data.drop('primary_department', axis=1, errors='ignore')
val_data = val_data.drop('primary_department', axis=1, errors='ignore')
test_data = test_data.drop('primary_department', axis=1, errors='ignore')

train_data = pd.get_dummies(train_data, columns=['primary_garment_group'], prefix='garment', dtype=int)
val_data = pd.get_dummies(val_data, columns=['primary_garment_group'], prefix='garment', dtype=int)
test_data = pd.get_dummies(test_data, columns=['primary_garment_group'], prefix='garment', dtype=int)

all_columns = set(train_data.columns).union(val_data.columns).union(test_data.columns)
print(all_columns)

train_data = train_data.reindex(columns=all_columns, fill_value=0)
val_data = val_data.reindex(columns=all_columns, fill_value=0)
test_data = test_data.reindex(columns=all_columns, fill_value=0)

{'purchased', 'garment_Accessories', 'product_price_std', 'customer_price_std', 'club_member_status_PRE-CREATE', 'max_price', 'garment_Dresses Ladies', 'age', 'garment_Dressed', 'garment_Skirts', 'sales_last_7_days', 'garment_Trousers Denim', 'num_purchases', 'garment_Jersey Fancy', 'avg_price', 'garment_Outdoor', 'garment_Woven/Jersey/Knitted mix Baby', 'garment_Shirts', 'customer_id', 'garment_Jersey Basic', 'avg_transaction_value', 'article_id', 'garment_Knitwear', 'garment_Trousers', 'min_price', 'category_diversity', 'garment_Shorts', 'garment_Dresses/Skirts girls', 'garment_Swimwear', 'FN', 'garment_Blouses', 'garment_Unknown', 'total_spent', 'fashion_news_frequency_REGULARLY', 'days_since_last_purchase', 'is_new_customer', 'garment_Under-, Nightwear', 'garment_Special Offers', 'garment_Shoes', 'avg_days_between_purchases', 'Active', 'sales_last_30_days', 'club_member_status_NOT_ACTIVE_MEMBER', 'garment_Socks and Tights', 'days_since_last_sale', 'days_since_first_sale'}


### X, y split
Outcome variable is `purchased` and 0 or 1 feature

In [10]:
X_train = train_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_train = train_data['purchased']
print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
X_val = val_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_val = val_data['purchased']
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
X_test = test_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_test = test_data['purchased']
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(284730, 43)
y_train.shape=(284730,)
X_val.shape=(440082, 43)
y_val.shape=(440082,)
X_test.shape=(326736, 43)
y_test.shape=(326736,)


## Save Processed Data
### as pickle files

In [11]:
# Save data as pickle to avoid reprocessing
processed_data_path = processed_path / 'product_recommendation'
processed_data_path.mkdir(parents=True, exist_ok=True)

with open(processed_data_path / 'X_train_base.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(processed_data_path / 'y_train_base.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(processed_data_path / 'X_val_base.pkl', 'wb') as f:
    pickle.dump(X_val, f)
with open(processed_data_path / 'y_val_base.pkl', 'wb') as f:
    pickle.dump(y_val, f)
with open(processed_data_path / 'X_test_base.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(processed_data_path / 'y_test_base.pkl', 'wb') as f:
    pickle.dump(y_test, f)